# Mini Project 2 — Part 2: Data Visualization

This notebook completes Part 2 using the `jmurph91_project_summary.csv` generated in Part 1.

For each assigned WoC project, this notebook:
- creates a complete monthly commit time series;
- includes zero-commit months;
- saves the monthly time series as a CSV;
- creates a 300-DPI line plot;
- finds the longest inactivity gap;
- counts inactivity gaps of at least three months;
- calculates commits after the longest gap;
- classifies the overall activity pattern; and
- creates the final project statistics CSV.

All generated per-project files are stored in the `part2_outputs` folder.

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

NETID = "jmurph91"

# Part 1 input file
INPUT_FILE = f"{NETID}_project_summary.csv"

# Put all Part 2 output files together
OUTPUT_DIR = Path("part2_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Input file: {INPUT_FILE}")
print(f"Output folder: {OUTPUT_DIR.resolve()}")

Input file: jmurph91_project_summary.csv
Output folder: /home/jmurph91/archeology/MiniProject2/MiniProject2/part2_outputs


In [11]:
# Load the Part 1 project summary CSV

df = pd.read_csv(
    INPUT_FILE,
    sep=";"
)

expected_columns = [
    "project_wocid",
    "commit_sha1",
    "author",
    "time",
    "commit message"
]

assert list(df.columns) == expected_columns, (
    f"Unexpected columns.\n"
    f"Expected: {expected_columns}\n"
    f"Found: {list(df.columns)}"
)

# Convert Unix timestamp to datetime
df["datetime"] = pd.to_datetime(
    df["time"],
    unit="s",
    errors="coerce"
)

assert df["datetime"].notna().all(), (
    "Some commit timestamps could not be converted."
)

# Create YYYY-MM month column
df["Month"] = df["datetime"].dt.to_period("M")

print(f"Loaded {len(df):,} commits.")
print(f"Projects: {df['project_wocid'].nunique()}")
print()
print(df["project_wocid"].value_counts().sort_index())

Loaded 50,009 commits.
Projects: 10

project_wocid
angel-ml_angel                         3418
bmclab_bmc                             2083
brial_brial                            2999
comit-network_xmr-btc-swap             4980
easystats_insight                      9905
jasp-stats_jasp-desktop               22023
juliarobotics_rigidbodydynamics.jl     2984
mmorise_world                           348
samreay_chainconsumer                   634
systempiper_systempipeshiny             635
Name: count, dtype: int64


In [12]:
def monthly_timeseries(project_df):
    """
    Create a monthly commit time series from the first
    month with activity through the last month with activity.
    
    Months with no commits are included with a value of 0.
    """
    
    counts = project_df.groupby("Month").size()

    first_month = counts.index.min()
    last_month = counts.index.max()

    # Every month between first and last commit
    all_months = pd.period_range(
        start=first_month,
        end=last_month,
        freq="M"
    )

    ts = (
        counts
        .reindex(all_months, fill_value=0)
        .rename_axis("Month")
        .reset_index(name="#Commits")
    )

    # Convert Period values to YYYY-MM strings
    ts["Month"] = ts["Month"].astype(str)

    ts["#Commits"] = ts["#Commits"].astype(int)

    return ts

In [13]:
def find_zero_gaps(ts):
    """
    Find every consecutive run of zero-commit months.
    """
    
    gaps = []
    start = None

    for i, row in ts.iterrows():

        month = row["Month"]
        is_zero = row["#Commits"] == 0

        # Beginning of a zero-commit period
        if is_zero and start is None:
            start = i

        # End of a zero-commit period
        if (not is_zero or i == len(ts) - 1) and start is not None:

            if is_zero and i == len(ts) - 1:
                end = i
            else:
                end = i - 1

            length = end - start + 1

            gaps.append({
                "start_index": start,
                "end_index": end,
                "start": ts.loc[start, "Month"],
                "end": ts.loc[end, "Month"],
                "length": length
            })

            start = None

    return gaps

In [14]:
def classify_activity(ts):
    """
    Classify the project's overall activity pattern.
    
    Categories:
    - steady
    - rising
    - declining
    - U-shaped
    - cyclical
    - irregular
    """

    values = ts["#Commits"].astype(float).to_numpy()

    n = len(values)

    if n < 3:
        return "irregular"

    # Linear trend
    x = np.arange(n)

    slope = np.polyfit(x, values, 1)[0]

    mean_value = values.mean()

    if mean_value == 0:
        return "irregular"

    normalized_slope = slope / mean_value

    # Divide timeline into thirds
    third = max(1, n // 3)

    early = values[:third].mean()
    middle = values[third:2 * third].mean()
    late = values[-third:].mean()

    # Avoid division problems
    if early == 0:
        early = 0.001

    # Coefficient of variation
    if mean_value != 0:
        cv = values.std() / mean_value
    else:
        cv = 0

    # Check for 12-month repetition
    lag12_corr = 0

    if n >= 36:
        first = values[:-12]
        second = values[12:]

        if np.std(first) > 0 and np.std(second) > 0:
            lag12_corr = np.corrcoef(first, second)[0, 1]

    # U-shaped pattern
    if (
        late > middle * 1.25
        and early > middle * 1.25
    ):
        return "U-shaped"

    # Cyclical / seasonal pattern
    if n >= 36 and lag12_corr >= 0.50:
        return "cyclical"

    # Rising trend
    if (
        normalized_slope >= 0.03
        and late > early * 1.20
    ):
        return "rising"

    # Declining trend
    if (
        normalized_slope <= -0.03
        and late < early * 0.80
    ):
        return "declining"

    # Steady activity
    if (
        abs(normalized_slope) < 0.03
        and cv < 1.0
    ):
        return "steady"

    # Anything else
    return "irregular"

In [15]:
projects = sorted(df["project_wocid"].unique())

timeseries_results = {}

for project in projects:

    print(f"Processing {project}...")

    project_df = df[
        df["project_wocid"] == project
    ].copy()

    # Build monthly time series
    ts = monthly_timeseries(project_df)

    timeseries_results[project] = ts

    # --------------------------------------------------------
    # Save monthly CSV
    # --------------------------------------------------------

    csv_path = (
        OUTPUT_DIR
        / f"{NETID}_commits_timeseries_{project}.csv"
    )

    ts.to_csv(
        csv_path,
        index=False,
        sep=";"
    )

    # --------------------------------------------------------
    # Create plot
    # --------------------------------------------------------

    plt.figure(figsize=(12, 6))

    plt.plot(
        ts["Month"],
        ts["#Commits"],
        marker="o",
        linestyle="-"
    )

    plt.xlabel("Month (YYYY-MM)")
    plt.ylabel("Number of Commits")
    plt.title(project)

    plt.grid(True)

    # Don't make the x-axis unreadable for long projects
    if len(ts) > 60:
        step = max(1, len(ts) // 20)
        tick_positions = range(0, len(ts), step)

        plt.xticks(
            tick_positions,
            ts["Month"].iloc[list(tick_positions)],
            rotation=45,
            ha="right"
        )
    else:
        plt.xticks(rotation=45, ha="right")

    plt.tight_layout()

    plot_path = (
        OUTPUT_DIR
        / f"{NETID}_timeseries_{project}.png"
    )

    plt.savefig(
        plot_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(
        f"  {ts['Month'].iloc[0]} -> "
        f"{ts['Month'].iloc[-1]}"
    )

print()
print(f"Finished {len(projects)} projects.")
print(f"All files saved in: {OUTPUT_DIR.resolve()}")

Processing angel-ml_angel...
  2017-05 -> 2025-10
Processing bmclab_bmc...
  2013-11 -> 2025-11
Processing brial_brial...
  2006-03 -> 2025-10
Processing comit-network_xmr-btc-swap...
  2020-09 -> 2025-08
Processing easystats_insight...
  2019-01 -> 2026-04
Processing jasp-stats_jasp-desktop...
  2013-06 -> 2026-04
Processing juliarobotics_rigidbodydynamics.jl...
  2016-04 -> 2025-03
Processing mmorise_world...
  2015-11 -> 2025-02
Processing samreay_chainconsumer...
  2016-07 -> 2026-04
Processing systempiper_systempipeshiny...
  2020-02 -> 2025-05

Finished 10 projects.
All files saved in: /home/jmurph91/archeology/MiniProject2/MiniProject2/part2_outputs


In [16]:
print("Files generated:")
print()

for file in sorted(OUTPUT_DIR.iterdir()):
    print(file.name)

Files generated:

jmurph91_commits_timeseries_angel-ml_angel.csv
jmurph91_commits_timeseries_bmclab_bmc.csv
jmurph91_commits_timeseries_brial_brial.csv
jmurph91_commits_timeseries_comit-network_xmr-btc-swap.csv
jmurph91_commits_timeseries_easystats_insight.csv
jmurph91_commits_timeseries_jasp-stats_jasp-desktop.csv
jmurph91_commits_timeseries_juliarobotics_rigidbodydynamics.jl.csv
jmurph91_commits_timeseries_mmorise_world.csv
jmurph91_commits_timeseries_samreay_chainconsumer.csv
jmurph91_commits_timeseries_systempiper_systempipeshiny.csv
jmurph91_part2_project_stats.csv
jmurph91_project_stats.csv
jmurph91_timeseries_angel-ml_angel.png
jmurph91_timeseries_bmclab_bmc.png
jmurph91_timeseries_brial_brial.png
jmurph91_timeseries_comit-network_xmr-btc-swap.png
jmurph91_timeseries_easystats_insight.png
jmurph91_timeseries_jasp-stats_jasp-desktop.png
jmurph91_timeseries_juliarobotics_rigidbodydynamics.jl.png
jmurph91_timeseries_mmorise_world.png
jmurph91_timeseries_samreay_chainconsumer.png
jm

In [17]:
# GitHub repository information collected for Part 1

github_info = pd.DataFrame([
    {
        "project": "comit-network_xmr-btc-swap",
        "number of stars": 712,
        "number of forks": 182,
        "last commit date": "2025-07-22",
    },
    {
        "project": "systempiper_systempipeshiny",
        "number of stars": 36,
        "number of forks": 6,
        "last commit date": "2025-05-20",
    },
    {
        "project": "mmorise_world",
        "number of stars": 1300,
        "number of forks": 264,
        "last commit date": "2025-02-21",
    },
    {
        "project": "easystats_insight",
        "number of stars": 442,
        "number of forks": 47,
        "last commit date": "2026-09-16",
    },
    {
        "project": "bmclab_bmc",
        "number of stars": 467,
        "number of forks": 264,
        "last commit date": "2026-09-13",
    },
    {
        "project": "brial_brial",
        "number of stars": 24,
        "number of forks": 20,
        "last commit date": "2026-09-01",
    },
    {
        "project": "juliarobotics_rigidbodydynamics.jl",
        "number of stars": 310,
        "number of forks": 53,
        "last commit date": "2024-11-07",
    },
    {
        "project": "angel-ml_angel",
        "number of stars": 6800,
        "number of forks": 1600,
        "last commit date": "2025-10-13",
    },
    {
        "project": "samreay_chainconsumer",
        "number of stars": 91,
        "number of forks": 21,
        "last commit date": "2026-04-06",
    },
    {
        "project": "jasp-stats_jasp-desktop",
        "number of stars": 1000,
        "number of forks": 231,
        "last commit date": "2026-09-25",
    },
])

github_info

,project,number of stars,number of forks,last commit date
0,comit-network_xmr-btc-swap,712,182,2025-07-22
1,systempiper_systempipeshiny,36,6,2025-05-20
2,mmorise_world,1300,264,2025-02-21
3,easystats_insight,442,47,2026-09-16
4,bmclab_bmc,467,264,2026-09-13
5,brial_brial,24,20,2026-09-01
6,juliarobotics_rigidbodydynamics.jl,310,53,2024-11-07
7,angel-ml_angel,6800,1600,2025-10-13
8,samreay_chainconsumer,91,21,2026-04-06
9,jasp-stats_jasp-desktop,1000,231,2026-09-25


In [18]:
project_records = []

for project in projects:

    project_df = df[
        df["project_wocid"] == project
    ].copy()

    ts = timeseries_results[project]

    # Find all zero-commit gaps
    gaps = find_zero_gaps(ts)

    # Only gaps of at least 3 months
    long_gaps = [
        gap for gap in gaps
        if gap["length"] >= 3
    ]

    # --------------------------------------------------------
    # Find longest inactivity gap
    # --------------------------------------------------------

    if gaps:

        # Longest gap
        # If tied, choose the earliest one
        longest_gap = sorted(
            gaps,
            key=lambda g: (-g["length"], g["start_index"])
        )[0]

        longest_gap_start = longest_gap["start"]
        longest_gap_end = longest_gap["end"]
        longest_gap_length = longest_gap["length"]

        # Number of commits after the end of the longest gap
        commits_after_gap = project_df[
            project_df["Month"]
            > pd.Period(longest_gap_end, freq="M")
        ].shape[0]

    else:

        longest_gap_start = ""
        longest_gap_end = ""
        longest_gap_length = 0
        commits_after_gap = 0

    # --------------------------------------------------------
    # Activity classification
    # --------------------------------------------------------

    activity_pattern = classify_activity(ts)

    # --------------------------------------------------------
    # GitHub information
    # --------------------------------------------------------

    github_row = github_info[
        github_info["project"] == project
    ]

    if len(github_row) == 1:

        github_row = github_row.iloc[0]

        stars = github_row["number of stars"]
        forks = github_row["number of forks"]
        last_commit_date = github_row["last commit date"]

    else:

        stars = np.nan
        forks = np.nan
        last_commit_date = ""

    # --------------------------------------------------------
    # Final project record
    # --------------------------------------------------------

    project_records.append({

        "project": project,

        "number of commits":
            len(project_df),

        "number of authors":
            project_df["author"].nunique(),

        "max time":
            project_df["time"].max(),

        "min time":
            project_df["time"].min(),

        "number of stars":
            stars,

        "number of forks":
            forks,

        "last commit date":
            last_commit_date,

        "LongestGapStart":
            longest_gap_start,

        "LongestGapEnd":
            longest_gap_end,

        "LongestGapLength":
            longest_gap_length,

        "NumCommitsAfterLastGap":
            commits_after_gap,

        "NumberOfGapsInTimeline":
            len(long_gaps),

        "ActivityPattern":
            activity_pattern
    })


project_stats = pd.DataFrame(project_records)

project_stats

,project,number of commits,number of authors,max time,min time,number of stars,number of forks,last commit date,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline,ActivityPattern
0,angel-ml_angel,3418,147,1760616761,1493817527,6800,1600,2025-10-13,2024-02,2025-05,16,54,2,declining
1,bmclab_bmc,2083,30,1762230919,1383617973,467,264,2026-09-13,2016-12,2017-03,4,1372,4,irregular
2,brial_brial,2999,34,1759360722,1141893962,24,20,2026-09-01,2013-02,2015-04,27,186,14,irregular
3,comit-network_xmr-btc-swap,4980,68,1755593438,1601272508,712,182,2025-07-22,,,0,0,0,U-shaped
4,easystats_insight,9905,86,1776168259,1548793879,442,47,2026-09-16,,,0,0,0,steady
5,jasp-stats_jasp-desktop,22023,242,1777093856,1370113512,1000,231,2026-09-25,2013-11,2013-11,1,21975,0,steady
6,juliarobotics_rigidbodydynamics.jl,2984,23,1740965887,1460686712,310,53,2024-11-07,2023-07,2024-09,15,27,4,cyclical
7,mmorise_world,348,46,1740126621,1447574109,1300,264,2025-02-21,2022-03,2023-05,15,11,9,declining
8,samreay_chainconsumer,634,18,1775513079,1468796482,91,21,2026-04-06,2022-10,2023-09,12,242,7,U-shaped
9,systempiper_systempipeshiny,635,20,1747714763,1582064135,36,6,2025-05-20,2023-11,2024-03,5,16,6,declining


In [19]:
stats_file = (
    OUTPUT_DIR
    / f"{NETID}_project_stats.csv"
)

project_stats.to_csv(
    stats_file,
    index=False,
    sep=";"
)

print("Project statistics saved.")
print(f"File: {stats_file}")
print(f"Projects: {len(project_stats)}")

Project statistics saved.
File: part2_outputs/jmurph91_project_stats.csv
Projects: 10


In [20]:
expected_stats_columns = [
    "project",
    "number of commits",
    "number of authors",
    "max time",
    "min time",
    "number of stars",
    "number of forks",
    "last commit date",
    "LongestGapStart",
    "LongestGapEnd",
    "LongestGapLength",
    "NumCommitsAfterLastGap",
    "NumberOfGapsInTimeline",
    "ActivityPattern"
]

assert list(project_stats.columns) == expected_stats_columns

assert len(project_stats) == 10

assert project_stats["project"].nunique() == 10

print("✓ Final project statistics verified")
print(f"✓ Rows: {len(project_stats)}")
print(f"✓ Columns: {len(project_stats.columns)}")
print()
print(project_stats.to_string(index=False))

✓ Final project statistics verified
✓ Rows: 10
✓ Columns: 14

                           project  number of commits  number of authors   max time   min time  number of stars  number of forks last commit date LongestGapStart LongestGapEnd  LongestGapLength  NumCommitsAfterLastGap  NumberOfGapsInTimeline ActivityPattern
                    angel-ml_angel               3418                147 1760616761 1493817527             6800             1600       2025-10-13         2024-02       2025-05                16                      54                       2       declining
                        bmclab_bmc               2083                 30 1762230919 1383617973              467              264       2026-09-13         2016-12       2017-03                 4                    1372                       4       irregular
                       brial_brial               2999                 34 1759360722 1141893962               24               20       2026-09-01         2013-02   

# Interpretation

The monthly commit time-series plots show how development activity changed over time for each project. The projects were classified as steady, rising, declining, U-shaped, cyclical, or irregular based on the overall patterns visible in their monthly activity.

A gap represents consecutive months with zero commits. The number of gaps reported below counts only gaps lasting at least three months.

In [21]:
def interpretation_text(row):

    project = row["project"]
    pattern = row["ActivityPattern"]

    gap_count = int(
        row["NumberOfGapsInTimeline"]
    )

    gap_length = int(
        row["LongestGapLength"]
    )

    if gap_length > 0:

        gap_description = (
            f"The longest period of inactivity lasted "
            f"{gap_length} month(s), from "
            f"{row['LongestGapStart']} through "
            f"{row['LongestGapEnd']}."
        )

    else:

        gap_description = (
            "No zero-commit inactivity gaps were found."
        )

    return (
        f"{project}: The overall activity pattern is "
        f"{pattern}. "
        f"{gap_description} "
        f"There were {gap_count} inactivity gap(s) "
        f"lasting at least three months. "
        f"After the end of the longest inactivity gap, "
        f"the project recorded "
        f"{int(row['NumCommitsAfterLastGap']):,} commits."
    )


for _, row in project_stats.iterrows():

    print(interpretation_text(row))
    print()

angel-ml_angel: The overall activity pattern is declining. The longest period of inactivity lasted 16 month(s), from 2024-02 through 2025-05. There were 2 inactivity gap(s) lasting at least three months. After the end of the longest inactivity gap, the project recorded 54 commits.

bmclab_bmc: The overall activity pattern is irregular. The longest period of inactivity lasted 4 month(s), from 2016-12 through 2017-03. There were 4 inactivity gap(s) lasting at least three months. After the end of the longest inactivity gap, the project recorded 1,372 commits.

brial_brial: The overall activity pattern is irregular. The longest period of inactivity lasted 27 month(s), from 2013-02 through 2015-04. There were 14 inactivity gap(s) lasting at least three months. After the end of the longest inactivity gap, the project recorded 186 commits.

comit-network_xmr-btc-swap: The overall activity pattern is U-shaped. No zero-commit inactivity gaps were found. There were 0 inactivity gap(s) lasting at

In [22]:
print("=" * 70)
print("PART 2 OUTPUT SUMMARY")
print("=" * 70)

print()
print(f"Input:")
print(f"  {INPUT_FILE}")

print()
print("Output folder:")
print(f"  {OUTPUT_DIR.resolve()}")

print()
print("Generated files:")

for file in sorted(OUTPUT_DIR.iterdir()):
    print(f"  ✓ {file.name}")

print()
print(f"Total output files: {len(list(OUTPUT_DIR.iterdir()))}")

PART 2 OUTPUT SUMMARY

Input:
  jmurph91_project_summary.csv

Output folder:
  /home/jmurph91/archeology/MiniProject2/MiniProject2/part2_outputs

Generated files:
  ✓ jmurph91_commits_timeseries_angel-ml_angel.csv
  ✓ jmurph91_commits_timeseries_bmclab_bmc.csv
  ✓ jmurph91_commits_timeseries_brial_brial.csv
  ✓ jmurph91_commits_timeseries_comit-network_xmr-btc-swap.csv
  ✓ jmurph91_commits_timeseries_easystats_insight.csv
  ✓ jmurph91_commits_timeseries_jasp-stats_jasp-desktop.csv
  ✓ jmurph91_commits_timeseries_juliarobotics_rigidbodydynamics.jl.csv
  ✓ jmurph91_commits_timeseries_mmorise_world.csv
  ✓ jmurph91_commits_timeseries_samreay_chainconsumer.csv
  ✓ jmurph91_commits_timeseries_systempiper_systempipeshiny.csv
  ✓ jmurph91_part2_project_stats.csv
  ✓ jmurph91_project_stats.csv
  ✓ jmurph91_timeseries_angel-ml_angel.png
  ✓ jmurph91_timeseries_bmclab_bmc.png
  ✓ jmurph91_timeseries_brial_brial.png
  ✓ jmurph91_timeseries_comit-network_xmr-btc-swap.png
  ✓ jmurph91_timeseries_ea

In [23]:
# Part 4 Reflection Files
# Writes all 10 project reflections to individual Markdown files
# while preserving the local notebook print output.

from pathlib import Path

OUTPUT_DIR = Path("part3_2_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


reflections = {
    "angel-ml_angel": """# Reflection: angel-ml_angel

Angel-ML/Angel has 3,418 commits from 147 authors, with activity from May 2017 through October 2025. Its longest inactivity gap lasted 16 months, from February 2024 through May 2025, and it had two gaps of at least three months. There were 54 commits after the longest gap, and the overall activity pattern was classified as declining.

Before the longest gap, the sampled commits focused on bug fixes and LCC/DeepWalk development. After the gap, activity included a DeepWalk example fix, a hash-partition fix, and several PSF-related features. The dominant themes therefore shifted from Other/Feature development to Feature development/Bug fixes.

The gap appears to represent a period of reduced or paused development followed by a targeted recovery rather than permanent abandonment. The post-gap commits also show a contributor shift, particularly involving rachelsun and rachelsunrh, who worked on several of the recovery changes. As of the September 30, 2025 cutoff, the project was Active, although the recovery did not restore the earlier level of activity.
""",

    "bmclab_bmc": """# Reflection: bmclab_bmc

BMC has 2,083 commits from 30 authors, with activity from November 2013 through November 2025. Its longest inactivity gap lasted four months, from December 2016 through March 2017, and there were four gaps of at least three months. The project had 1,372 commits after the longest gap and was classified as having an irregular activity pattern.

The sampled commits before and after the longest gap were both primarily classified as Other/Other. The commit messages did not provide enough evidence to identify a specific reason for the interruption.

The gap appears to have been a temporary pause rather than abandonment. The project recovered strongly after the gap and continued to accumulate a large number of commits. As of September 30, 2025, the project was Active, although its overall timeline remained irregular.
""",

    "brial_brial": """# Reflection: brial_brial

BRiAl has 2,999 commits from 34 authors, with activity from March 2006 through October 2025. Its longest inactivity gap was 27 months, from February 2013 through April 2015, and the project had 14 gaps of at least three months. It had 186 commits after the longest gap and was classified as irregular.

Before the longest gap, the sampled commits were dominated by Other and Release activity. After the gap, the sampled commits included Feature development and Other activity. The long interruption was followed by a clear return to development, but the available commit messages did not provide enough evidence to confidently determine the cause of the gap.

The project therefore appears to have experienced a long interruption without being permanently abandoned. It recovered and continued developing for many years afterward. As of September 30, 2025, BRiAl was Active.
""",

    "comit-network_xmr-btc-swap": """# Reflection: comit-network_xmr-btc-swap

COMIT Network's XMR-BTC swap project has 4,980 commits from 68 authors, with activity from September 2020 through August 2025. The project did not have a zero-commit inactivity gap, so there was no meaningful longest gap requiring before-and-after analysis. Its activity pattern was classified as U-shaped.

Because there was no extended inactivity period, the project does not have a corresponding before/after gap theme comparison. The commit history instead shows continuous development involving features, maintenance, dependency changes, bug fixes, and automated contributions. Examples include transaction watching and logging changes, build-related work, and release activity.

The continuous timeline suggests that the project remained under active development rather than experiencing a prolonged abandonment period. As of September 30, 2025, the project was Active.
""",

    "easystats_insight": """# Reflection: easystats_insight

easystats/insight has 9,905 commits from 86 authors, with activity from January 2019 through April 2026. There was no zero-commit inactivity gap in the timeline, and the project was classified as having a steady activity pattern.

Because there was no extended inactivity period, there was no longest gap requiring before-and-after commit analysis. The commit history instead demonstrates sustained development across the project's lifetime.

The absence of a significant inactivity gap suggests consistent maintenance and development rather than periods of abandonment. For the required September 30, 2025 status cutoff, the project was Active; later commits in 2026 were not used to determine that status.
""",

    "jasp-stats_jasp-desktop": """# Reflection: jasp-stats_jasp-desktop

JASP has 22,023 commits from 242 authors, with activity from June 2013 through April 2026. Its timeline contained only a one-month gap around November 2013, which does not qualify as a gap of at least three months. The project was classified as having a steady activity pattern and had more than 21,000 commits after that early gap.

The sampled commits around the early gap were dominated by bug fixes both before and after the gap. Because the interruption was so short, it appears to have been a brief fluctuation rather than a meaningful development pause.

JASP provides a strong contrast with projects such as BRiAl and Angel, where development stopped for much longer periods. Its large number of commits and continued activity indicate a consistently maintained project. As of September 30, 2025, JASP was Active.
""",

    "juliarobotics_rigidbodydynamics.jl": """# Reflection: juliarobotics_rigidbodydynamics.jl

RigidBodyDynamics.jl has 2,984 commits from 23 authors, with activity from April 2016 through March 2025. Its longest inactivity gap lasted 15 months, from July 2023 through September 2024, and there were four gaps of at least three months. Only 27 commits occurred after the longest gap, and the overall activity pattern was classified as cyclical.

The sampled commits before and after the longest gap were primarily dependency updates. The commit history did not provide enough evidence to identify a specific reason for the long interruption.

Although the project showed some recovery after the gap, the recovery was limited compared with its earlier history. Most importantly, historical recovery does not necessarily mean that a project is currently healthy. Under the required September 30, 2025 cutoff, the latest qualifying commit was more than six months old, so the project was classified as Inactive.
""",

    "mmorise_world": """# Reflection: mmorise_world

World has 348 commits from 46 authors, with activity from November 2015 through February 2025. Its longest inactivity gap lasted 15 months, from March 2022 through May 2023, and there were nine gaps of at least three months. Only 11 commits occurred after the longest gap, and its activity pattern was classified as declining.

Before the longest gap, the sampled commits were primarily Feature development and Other activity. After the gap, the themes shifted toward Dependency updates and Bug fixes.

The project therefore appears to have experienced a weak maintenance-oriented recovery rather than a return to its earlier development level. The relatively small number of post-gap commits supports this interpretation. Because its latest commit before the September 30, 2025 cutoff was earlier than April 1, 2025, the project was classified as Inactive.
""",

    "samreay_chainconsumer": """# Reflection: samreay_chainconsumer

ChainConsumer has 634 commits from 18 authors, with activity from July 2016 through April 2026. Its longest inactivity gap lasted 12 months, from October 2022 through September 2023, and there were seven gaps of at least three months. There were 242 commits after the longest gap, producing a U-shaped activity pattern.

The sampled commits before and after the longest gap were dominated by Other and Bug fixes. This suggests that the recovery focused largely on maintenance, corrections, and continued refinement rather than a clearly identifiable new development phase.

The available commit messages did not point to one specific reason for the interruption. However, the large number of commits after the gap demonstrates a strong recovery. As of September 30, 2025, ChainConsumer was Active.
""",

    "systempiper_systempipeshiny": """# Reflection: systemPipeR/systemPipeShiny

systemPipeShiny has 635 commits from 20 authors, with activity from February 2020 through May 2025. Its longest inactivity gap lasted five months, from November 2023 through March 2024, and there were six gaps of at least three months. There were 16 commits after the longest gap, and the activity pattern was classified as declining.

Both the pre-gap and post-gap samples were dominated by Dependency updates and Bug fixes. This indicates that the recovery was primarily maintenance-oriented rather than a major return to feature development.

The project appears to have recovered temporarily after the longest gap, but its activity did not return to a strong sustained development pattern. Since its latest commit was in May 2025, which is more than six months before the September 30, 2025 cutoff, the project was classified as Inactive.
"""
}

# Write and print every reflection
for project, reflection in reflections.items():
    output_file = OUTPUT_DIR / f"{NETID}_reflection_{project}.md"
    
    output_file.write_text(reflection.strip() + "\n", encoding="utf-8")
    
    print("=" * 80)
    print(f"FILE: {output_file}")
    print("=" * 80)
    print(reflection)
    print()

print(f"Successfully wrote {len(reflections)} reflection files to {OUTPUT_DIR}/")

FILE: part3_2_outputs/jmurph91_reflection_angel-ml_angel.md
# Reflection: angel-ml_angel

Angel-ML/Angel has 3,418 commits from 147 authors, with activity from May 2017 through October 2025. Its longest inactivity gap lasted 16 months, from February 2024 through May 2025, and it had two gaps of at least three months. There were 54 commits after the longest gap, and the overall activity pattern was classified as declining.

Before the longest gap, the sampled commits focused on bug fixes and LCC/DeepWalk development. After the gap, activity included a DeepWalk example fix, a hash-partition fix, and several PSF-related features. The dominant themes therefore shifted from Other/Feature development to Feature development/Bug fixes.

The gap appears to represent a period of reduced or paused development followed by a targeted recovery rather than permanent abandonment. The post-gap commits also show a contributor shift, particularly involving rachelsun and rachelsunrh, who worked on several